<a href="https://colab.research.google.com/github/DimiGretsistas/Automated-Customers-Reviews-Project/blob/main/notebooks/05_gpt_product_recommendation_generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GPT Generator

## Goal
Use an OpenAI GPT model to generate product recommendation articles from the real cleaned Amazon review dataset.

This notebook does not use LoRA. It loads the dataset, extracts product insights, and sends structured review information to GPT.

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/clean_reviews.csv",
    engine="python",
    on_bad_lines="skip"
)

df.head()

,name,reviews.rating,reviews.text,sentiment,clean_review
0,AmazonBasics AAA Performance Alkaline Batterie...,3,I order 3 of them and one of the item is bad q...,neutral,i order 3 of them and one of the item is bad q...
1,AmazonBasics AAA Performance Alkaline Batterie...,4,Bulk is always the less expensive way to go fo...,positive,bulk is always the less expensive way to go fo...
2,AmazonBasics AAA Performance Alkaline Batterie...,5,Well they are not Duracell but for the price i...,positive,well they are not duracell but for the price i...
3,AmazonBasics AAA Performance Alkaline Batterie...,5,Seem to work as well as name brand batteries a...,positive,seem to work as well as name brand batteries a...
4,AmazonBasics AAA Performance Alkaline Batterie...,5,These batteries are very long lasting the pric...,positive,these batteries are very long lasting the pric...


## Create Review Sample

Use a smaller sample to keep the experiment fast and lightweight.

In [ ]:
#Creating a random sample of 1500 rows from the dataframe
sample_df = df.sample(n=1500, random_state=42).copy()

#Removing line breaks and commas from the "name" column, then trim spaces
sample_df["name"] = sample_df["name"].str.replace(r"[\r\n,]+", " ", regex=True).str.strip()

#Displaying the shape (rows, columns) of the sampled dataframe
sample_df.shape

(1500, 5)

## Create Embeddings and Clusters

Convert reviews into embeddings and group them into product-related clusters.

In [ ]:
!pip install sentence-transformers -q
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

#Converting the "clean_review" column into a Python list
texts = sample_df["clean_review"].tolist()

#Loading the pre-trained embedding model
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

#Generating embeddings for all review texts
embeddings = embedding_model.encode(texts, show_progress_bar=True)

#Creating a KMeans clustering model with 5 clusters
kmeans = KMeans(n_clusters=5, random_state=42)

#Assigning cluster labels to each review
sample_df["cluster"] = kmeans.fit_predict(embeddings)

#Displaying the first rows with business names and assigned clusters
sample_df[["name", "cluster"]].head()

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/47 [00:00<?, ?it/s]

,name,cluster
45193,Amazon Kindle Paperwhite - eBook reader - 4 GB...,2
22309,Fire Kids Edition Tablet 7 Display Wi-Fi 16...,0
34885,Fire Tablet 7 Display Wi-Fi 8 GB - Includes...,0
16409,All-New Fire HD 8 Tablet 8 HD Display Wi-Fi ...,2
23893,Kindle E-reader - White 6 Glare-Free Touchscr...,0


## Prepare Product Insights

Extract top products, weakest product, and customer complaints from each cluster.

In [ ]:
def prepare_cluster_insights(cluster_df):
    product_stats = cluster_df.groupby("name").agg(
        avg_rating=("reviews.rating", "mean"),
        review_count=("reviews.rating", "count")
    ).reset_index()

    product_stats = product_stats[product_stats["review_count"] >= 3]

    top_products = product_stats.sort_values(
        by="avg_rating",
        ascending=False
    ).head(3)

    worst_product = product_stats.sort_values(
        by="avg_rating",
        ascending=True
    ).head(1)

    complaints = cluster_df[
        cluster_df["sentiment"] == "negative"
    ]["clean_review"].head(5).tolist()

    return top_products, worst_product, complaints

## Load OpenAI GPT Model

Load the OpenAI client using the API key stored safely in Colab secrets.

In [ ]:
!pip install openai -q

from google.colab import userdata
from openai import OpenAI

client = OpenAI(
    api_key=userdata.get("oAI_API")
)

## Generate Product Recommendation Article

Use GPT to generate a short recommendation article from real cluster insights.

In [ ]:
cluster_id = 1

cluster_df = sample_df[sample_df["cluster"] == cluster_id]

top_products, worst_product, complaints = prepare_cluster_insights(cluster_df)

prompt = f"""
You are a business-focused tech product reviewer.

Write a short product recommendation article based ONLY on the structured review data below.

Cluster ID:
{cluster_id}

Top products:
{top_products.to_string(index=False)}

Weakest product:
{worst_product.to_string(index=False)}

Customer complaints:
{complaints}

Rules:
- Only use the listed products and complaints
- Do not invent products, features, prices, or brands
- Discuss the top 3 products from the provided data
- Explain when a customer should choose one top product over another
- Explain key differences using only rating and review information
- Summarize the main complaints
- Mention the weakest product and why it should be avoided
- Finish with a clear final recommendation
- Write in a short business-review style similar to a product recommendation website
- Do not repeat ratings excessively
- Avoid long introductions
- Keep it under 150 words
- Use simple, professional language
"""

response = client.responses.create(
    model="gpt-4.1-mini",
    input=prompt
)

gpt_generated_text = response.output_text

print(gpt_generated_text)

Among Amazon’s Alexa-enabled devices, the Echo Plus with Built-In Hub leads with the highest rating and strong customer approval, ideal for users wanting a reliable smart home central hub. The Amazon Tap Smart Assistant offers excellent ratings and suits those seeking a portable, voice-controlled speaker. The Echo (White) is well-rated and popular, making it a solid all-around choice for home use.

However, some customers report voice recognition and music playback issues, particularly with artist requests and inconsistent service performance. These complaints highlight the need for careful selection based on user expectations.

The lowest-rated product, the Amazon Tap - Alexa-Enabled Portable Bluetooth Speaker, should be avoided due to its lower satisfaction and recurring complaints about voice recognition and music handling.

For best reliability and versatility, the Echo Plus stands out as the top recommendation. Choose it for a smart home hub, or the Tap Smart Assistant for portabl

# AI-Generated Product Recommendation Articles

These recommendation articles were generated using NLP techniques including sentiment classification, clustering, prompt engineering, and generative AI models.  
(Note: clusters overlap semantically.)

---

# Cluster 0 — Family Tablets & Entertainment Devices

## Top Products

- Amazon Tap Smart Assistant Alexa-enabled (Black)
- All-New Fire HD 8 Tablet
- Fire Kids Edition Tablet


## Generated Recommendation Article
Among Amazon’s top-rated family and entertainment devices, the Tap Smart Assistant Alexa-enabled (black) stands out with a perfect rating and is best suited for users looking for a reliable voice-controlled smart assistant. The All-New Fire HD 8 Tablet combines strong display quality and high customer satisfaction, making it a good choice for general media consumption and everyday tablet use. The Fire Kids Edition Tablet offers a durable kid-proof design and received strong feedback from families, making it the best option for children.

Customers reported issues such as devices failing early, misleading feature descriptions, and poor accessory quality. These complaints mainly affected lower-rated products and raised concerns about durability and reliability.

The weakest product in this cluster was the Brand New Amazon Kindle Fire 16GB 7" IPS Display Tablet, which received noticeably weaker feedback and should be avoided due to reliability concerns.

For general tablet use, the Fire HD 8 Tablet is the strongest overall choice, while the Fire Kids Edition Tablet is recommended for families and younger users.



---

# Cluster 1 — Smart Speakers / Alexa Devices

## Top Products

- Echo Plus with Built-In Hub (Silver)
- Amazon Tap Smart Assistant Alexa-enabled Speaker
- Echo (White)

## Generated Recommendation Article

Among Amazon smart speaker devices, the Echo Plus with Built-In Hub (Silver) achieved the highest performance with a 4.92 rating from 12 reviews. The Amazon Tap Smart Assistant Alexa-enabled speaker followed closely with a 4.75 rating, while the standard Echo (White) maintained strong customer satisfaction with a 4.72 rating across 36 reviews.

Customer complaints mainly focused on voice recognition problems and inconsistent music playback behavior. Some users reported incorrect Alexa responses and unreliable speaker performance.

The Amazon Tap Portable Bluetooth Speaker received the weakest feedback with a 4.0 rating from 3 reviews due to voice detection and playback issues.

For smart home usage and voice assistant functionality, the Echo Plus with Built-In Hub is the strongest recommendation in this cluster.

---

# Cluster 2 — Premium Reading & Entertainment Devices

## Top Products

- Kindle Oasis E-reader with Leather Charging Cover
- Fire Kids Edition Tablet
- Echo (White)

## Generated Recommendation Article

The Kindle Oasis E-reader with Leather Charging Cover, Fire Kids Edition Tablet, and Echo (White) each achieved perfect 5.0 ratings from multiple reviews, demonstrating strong customer satisfaction across reading, entertainment, and smart home usage.

However, the All-New Fire HD 8 Tablet received weaker feedback with a 4.13 rating from 8 reviews. Customers reported navigation difficulties, weaker speaker quality compared to older models, intrusive game pop-ups, unstable SD card functionality, and repeated factory reset requirements.

These issues negatively affected the overall user experience, especially for customers seeking a smooth reading or entertainment device.

For reliable performance and higher customer satisfaction, the Kindle Oasis and Fire Kids Edition Tablet are the best recommendations in this cluster.

---

# Cluster 3 — Batteries / Power Products

## Top Products

- AmazonBasics AA Performance Alkaline Batteries (48 Count)
- AmazonBasics AAA Performance Alkaline Batteries (36 Count)

## Generated Recommendation Article

Among Amazon power products, the AmazonBasics AA Performance Alkaline Batteries (48 Count) performed best with a 4.38 rating from 53 reviews, showing reliable battery performance across multiple devices. The AmazonBasics AAA Performance Alkaline Batteries received a lower 3.86 rating from 123 reviews, indicating more inconsistent customer experiences.

The main complaints focused on poor battery life, fast power loss, and defective battery batches. Several customers reported needing frequent battery replacements, especially with AAA batteries.

The Fire Tablet 7" Display Wi-Fi 8 GB (Magenta) received the weakest overall feedback with a 3.0 rating from only 3 reviews.

For dependable battery performance, the AmazonBasics AA batteries are the strongest recommendation in this cluster, while caution is advised for the AAA battery models.

---

# Cluster 4 — Budget Tablets / Kids Devices

## Top Products

- All-New Fire HD 8 Kids Edition Tablet
- Amazon Kindle Paperwhite
- Fire Tablet (7" Display, 16 GB)

## Generated Recommendation Article

Among Amazon budget tablets and kids devices, the All-New Fire HD 8 Kids Edition Tablet achieved a perfect 5.0 rating from 4 reviews and was praised for its durable kid-proof case and display quality. The Amazon Kindle Paperwhite also performed strongly with a 4.8 average rating from 10 reviews, making it a reliable option for eBook reading. The Fire Tablet (7" Display, 16 GB) maintained solid customer satisfaction with a 4.76 rating from 17 reviews.

The weakest product in this cluster was the All-New Fire 7 Tablet with Alexa, which received a 3.67 rating from 3 reviews. Customers reported issues such as headphone jack defects, poor battery life, slow performance, limited memory, and lack of screen-sharing compatibility.

For reliable everyday use, the Fire HD 8 Kids Edition Tablet and Kindle Paperwhite are the strongest recommendations in this cluster.

# Final Conclusions

This project successfully demonstrates a complete NLP pipeline for analyzing and summarizing Amazon product reviews using modern machine learning and generative AI techniques.

The sentiment classification stage achieved strong performance using DistilBERT, with approximately 95% accuracy. The model performed well overall, although neutral reviews remained more difficult to classify due to dataset imbalance.

The clustering stage combined MiniLM sentence embeddings with KMeans clustering to group semantically similar reviews and products. While the silhouette score was relatively low, manual inspection confirmed that the generated clusters captured meaningful themes such as smart speakers, tablets, batteries, and reading devices.

Several summarization and text generation approaches were explored throughout the project, including:
- Zero-shot prompting
- 1-shot prompting
- 2-shot prompting
- Full fine-tuning with FLAN-T5
- LoRA fine-tuning
- GPT-based generation

Prompt engineering significantly improved output quality. The 2-shot prompting approach generated more structured and reliable recommendation articles than zero-shot prompting and reduced hallucinations and irrelevant outputs.

The full fine-tuning and LoRA experiments demonstrated the limitations of training generative models on very small datasets. Both approaches often copied noisy review text or product table fragments instead of generating clean recommendation summaries. ROUGE and BLEU evaluation metrics confirmed the weak performance of the fine-tuned models.

The strongest overall generation quality was achieved using GPT-based generation with structured prompts. Compared to the locally fine-tuned models, GPT produced more coherent, concise, and business-oriented recommendation articles with fewer hallucinations and better handling of customer complaints and product comparisons.

The project highlights several important lessons:
- High-quality prompts strongly influence generation quality
- Small datasets are insufficient for reliable generative fine-tuning
- Manual inspection is essential alongside automatic metrics
- Larger language models can significantly improve summarization quality without additional fine-tuning

Overall, the project successfully transformed raw customer reviews into structured business insights, recommendation summaries, and product-focused articles using modern NLP and generative AI techniques.